# 🏗️ Notebook 1: LinkedIn Connections — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/linkedin-connections
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A social graph service. Users send/accept connection requests, see their 1st/2nd/3rd-degree network, and get **People You May Know** suggestions.

The graph is the whole product — everything interesting is a graph algorithm at scale.

## Requirements

### Functional
- Send/accept/reject connection requests.
- List 1st-degree connections.
- Compute degrees of separation.
- Suggest People You May Know.

### Non-functional
- 1st-degree lookup: O(1) / <50ms.
- PYMK can be precomputed (daily) — latency tolerant.

## Back-of-envelope

- 1B users, avg 500 connections → 500B edges (undirected).
- Power-law distribution: top users have 30k+ connections (hot nodes).

## High-level architecture

```
  [API] ──► Graph Service
              │
      ┌───────┼─────────┐
      ▼       ▼         ▼
    Edge DB  Cache    PYMK cache
   (sharded) (Redis)  (precomputed)
              │
              ▼
        Offline pipeline (Spark) → PYMK
```

- Edges sharded by `user_id`.
- Hot users' edge lists cached with TTL.

## Why these choices?

- Each service in the diagram owns one responsibility — easier to scale and reason about.
- Stateless services scale horizontally; stateful stores are chosen per access pattern.
- The next two notebooks zoom into the **data model + APIs** and one **deep-dive algorithm**.